In [1]:
import os
import xarray as xr
import rioxarray
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box

In [2]:
# Define file paths
era5_dir = "/global/scratch/users/yougsanghvi/era5_hourly_by_year/"  # Each GRIB per year
gdnat_dir = "/global/scratch/users/yougsanghvi/gdnat_tiff_files_by_yr/"     # Each TIFF per year

years = list(range(2000, 2005))  # 2000 to 2004

# Bounding box for mainland USA (approx)
usa_bounds = {
    "min_lon": -125,
    "max_lon": -66,
    "min_lat": 24,
    "max_lat": 50
}


In [4]:
era5_daily = []

for year in years:
    ds = xr.open_dataset(os.path.join(era5_dir, f"era5_data_{year}.grib"), engine="cfgrib")

    # Clip to mainland USA
    ds = ds.sel(latitude=slice(usa_bounds["max_lat"], usa_bounds["min_lat"]),
                longitude=slice(usa_bounds["min_lon"], usa_bounds["max_lon"]))

    # Resample to daily average
    ds_daily = ds.resample(time="1D").mean()

    # Add time-based columns
    ds_daily["year"] = ds_daily["time.year"]
    ds_daily["month"] = ds_daily["time.month"]
    ds_daily["day"] = ds_daily["time.day"]

    era5_daily.append(ds_daily)

era5_combined = xr.concat(era5_daily, dim="time")


/global/home/users/yougsanghvi/global_suicide_dummy/climate-env/lib64/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
/global/home/users/yougsanghvi/global_suicide_dummy/climate-env/lib64/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
/global/home/users/yougsanghvi/global_suicide_dummy/climate-env/lib64/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. 

In [5]:
gdnat_list = []

for year in years:
    gdnat_path = os.path.join(gdnat_dir, f"gdnat_{year}.tif")
    gdnat_ds = rioxarray.open_rasterio(gdnat_path)
    
    # Clip to mainland USA
    gdnat_ds = gdnat_ds.rio.clip_box(**usa_bounds)
    
    gdnat_ds = gdnat_ds.squeeze()  # remove band dimension
    gdnat_ds = gdnat_ds.expand_dims(time=[pd.Timestamp(f"{year}-01-01")])  # dummy time

    gdnat_list.append(gdnat_ds)

gdnat_combined = xr.concat(gdnat_list, dim="time")


TypeError: RasterArray.clip_box() got an unexpected keyword argument 'min_lon'

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
cc_mean.plot(ax=ax, cmap='coolwarm', cbar_kwargs={'label': 'ERA5 - GDNat Temp (°C)'})
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.set_title("Mean Temperature Difference (ERA5 - GDNat, 2000–2004)")
plt.show()
